<a href="https://colab.research.google.com/github/Zhengdao-Lu/roblem-Set-3-Econ_3916_PS3.ipynb/blob/main/ps3_starter_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [ ]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [ ]:
N_CUSTOMERS = 4_000

N_WEST = 2_800
regions = np.array(["West"] * N_WEST + ["East"] * (N_CUSTOMERS - N_WEST))
rng.shuffle(regions)
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, N_CUSTOMERS)

customers_df = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "region": regions,
    "revenue": np.maximum(
        REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, N_CUSTOMERS),
        REV_FLOOR,
    ).round(2),
    "signup_date": pd.Timestamp("2022-01-01")
                   + pd.to_timedelta(rng.integers(0, 1095, N_CUSTOMERS), unit="D"),
})
orders_df = pd.DataFrame({
    "customer_id": np.repeat(customers_df["customer_id"], n_orders),
})
orders_df.insert(0, "order_id", np.arange(1, len(orders_df) + 1))

con.register("customers_df", customers_df)
con.register("orders_df", orders_df)
con.execute("CREATE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE TABLE orders AS SELECT * FROM orders_df")
print(con.sql("SELECT COUNT(*) AS customers FROM customers").df())
print(con.sql("SELECT COUNT(*) AS west FROM customers WHERE region = 'West'").df())
print(con.sql("SELECT COUNT(*) AS orders FROM orders").df())

   customers
0       4000
   west
0  2800
   orders
0   27211


### Step 1.2: Write both queries (7 pts)

In [ ]:
low_sql = """
    SELECT COUNT(*) AS rows_used, ROUND(AVG(revenue), 2) AS average_revenue
    FROM customers
    WHERE region = 'West'
"""

high_sql = """
    SELECT COUNT(*) AS rows_used, ROUND(AVG(c.revenue), 2) AS average_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

low = con.sql(low_sql).df().iloc[0]
high = con.sql(high_sql).df().iloc[0]
ratio = high["average_revenue"] / low["average_revenue"]
print(f"Per customer: ${low['average_revenue']:,.2f} over {int(low['rows_used']):,} rows")
print(f"After join:  ${high['average_revenue']:,.2f} over {int(high['rows_used']):,} rows")
print(f"Ratio: {ratio:.3f}")

assert 780 <= low["average_revenue"] <= 845
assert 1160 <= high["average_revenue"] <= 1320
assert 1.45 <= ratio <= 1.60

Per customer: $811.17 over 2,800 rows
After join:  $1,231.82 over 19,082 rows
Ratio: 1.519


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

*Your answer here.*

Finance's lower number is the right one for average revenue per customer. In this synthetic warehouses the unjoined query giver 811.17 over 2800 West customers because each customer has exactly one vote. THe joined query giver $1231.82 over 19082 order rowas. This is artithmaticly valid, but its unit does not match the lable.

### Step 1.3: The rule (10 pts)

In [ ]:
def check_join(join_sql):
    before = con.sql("SELECT COUNT(*) FROM customers WHERE region = 'West'").fetchall()[0][0]
    after = con.sql(f"SELECT COUNT(*) FROM ({join_sql}) AS joined_rows").fetchall()[0][0]
    if after != before:
        print(f"Warning: {before:,} rows before the join, {after:,} after")

bad_join = """
    SELECT c.customer_id, c.revenue
    FROM customers c JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

good_join = """
    SELECT c.customer_id, c.revenue
    FROM customers c
    LEFT JOIN (SELECT customer_id, COUNT(*) AS n_orders
               FROM orders GROUP BY customer_id) o
    ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

print("Bad join:")
check_join(bad_join)
print("Good join:")
check_join(good_join)

Bad join:
Good join:


*Your answer here.*



## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [ ]:
missing_ids = rng.choice(customers_df["customer_id"], size=600, replace=False)
con.register("missing_ids", pd.DataFrame({"customer_id": missing_ids}))
con.execute("""
    UPDATE customers SET region = NULL
    WHERE customer_id IN (SELECT customer_id FROM missing_ids)
""")
missing = con.sql("SELECT COUNT(*) FROM customers WHERE region IS NULL").fetchall()[0][0]
print(f"Missing region: {missing:,} of {N_CUSTOMERS:,} customers")

Missing region: 600 of 4,000 customers


### Step 2.2: Lose them, then find them (15 pts)

In [ ]:
lost_query = "SELECT COUNT(*) FROM customers WHERE region != 'West'"
kept_query = "SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'"
lost_count = con.sql(lost_query).fetchall()[0][0]
kept_count = con.sql(kept_query).fetchall()[0][0]
print(f"region != 'West': {lost_count:,} rows")
print(f"Including unknown regions: {kept_count:,} rows")
print(f"Difference: {kept_count - lost_count:,} rows")

region != 'West': 1,046 rows
Including unknown regions: 1,646 rows
Difference: 600 rows


*Your answer here.*

region != 'West' retuned 1046 rows, region IS DISTINCT FROM 'West' retuned 1646. region != 'West' evaluate to unknown for NULL regions, and WHERE retains only true conditoons. If mission-region customers are systematically newer, smaller, or more likely to chur, the first query selects a different population and can make ovserved regional revenue look hgher and chrum look lower than they really are.

### Step 2.3: Aggregate without collapsing (10 pts)

In [ ]:
window_sql = """
    SELECT customer_id, region, revenue,
           ROUND(AVG(revenue) OVER (PARTITION BY region), 2) AS region_avg,
           ROUND(revenue - AVG(revenue) OVER (PARTITION BY region), 2) AS vs_region
    FROM customers
    ORDER BY customer_id
"""
with_average = con.sql(window_sql).df()
print(with_average.head(8).to_string(index=False))
print(f"Rows before: {N_CUSTOMERS:,}; rows after: {len(with_average):,}")
assert len(with_average) == N_CUSTOMERS

 customer_id region  revenue  region_avg  vs_region
           1   West   760.63      810.33     -49.70
           2   West   736.81      810.33     -73.52
           3   West   713.32      810.33     -97.01
           4   None   525.75      813.04    -287.29
           5   West   837.24      810.33      26.91
           6   West  1260.78      810.33     450.45
           7   West   879.82      810.33      69.49
           8   East   385.74      804.97    -419.23
Rows before: 4,000; rows after: 4,000


## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [ ]:
causal_rng = np.random.default_rng(2026)
n = 20_000
size = np.maximum(causal_rng.normal(100, 20, n), 10)
chance_west = 1 / (1 + np.exp(-(size - 100) / 15))
west = causal_rng.binomial(1, chance_west, n)
TRUE_EFFECT = 25
revenue = 400 + 8 * size + TRUE_EFFECT * west + causal_rng.normal(0, 100, n)

programme_df = pd.DataFrame({"west": west, "size": size, "revenue": revenue})
naive = programme_df.loc[west == 1, "revenue"].mean() - programme_df.loc[west == 0, "revenue"].mean()
adjusted = smf.ols("revenue ~ west + size", data=programme_df).fit().params["west"]
print(f"Truth: ${TRUE_EFFECT:.2f}")
print(f"Naive West–East gap: ${naive:.2f}")
print(f"Adjusting for existing size: ${adjusted:.2f}")

Truth: $25.00
Naive West–East gap: $185.59
Adjusting for existing size: $24.06


*Your answer here.*

In the simulation, the true program effect is 25. The vaice West-East gap is 185.59 because West customers already had larger accounts, which independently raise renvuen. In the actual warehouse, the difference alone cannot establish causation, observational adjustment is credible only if the important common causes are measured and modeled approprately.

### Step 3.2: The control that ruins it (13 pts)

In [ ]:
average_tickets = np.maximum(0.2, 8 + 3 * west + 0.015 * (revenue - 1200))
programme_df["tickets"] = causal_rng.poisson(average_tickets)
with_tickets = smf.ols("revenue ~ west + size + tickets", data=programme_df).fit().params["west"]
print(f"Truth: ${TRUE_EFFECT:.2f}")
print(f"Naive gap: ${naive:.2f}")
print(f"Adjusting for size: ${adjusted:.2f}")
print(f"Also adjusting for tickets: ${with_tickets:.2f}")

Truth: $25.00
Naive gap: $185.59
Adjusting for size: $24.06
Also adjusting for tickets: $-18.31


*Your answer here.*

West customers already had larger accounts before the programme, their high revenue therefore cannnot be credited to account management. In a simulation where the tru grogramme effect is 25, acounting for existing size moves the estimate from the raw 185.59 gap to 24.06.Comparing customer with the same ticket count fistorts the comparison and moves the estimate to -18.31, farther from the truth. For the real rollout, I would use pre-programme data and comparison design, to randomized assignment before presenting the causal claim.




## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [ ]:
def review_query(con, sql):
    """The model's function from the P.R.I.M.E. prompt, then your corrections:
    returns a list of structured warnings for one SELECT."""
    ...

In [ ]:
# Run review_query on your Phase 1 and Phase 2 queries.

#Phase 1 query

"""Review simple DuckDB SELECTs over customers and orders.

Supported SQL: one plain base table, optionally one equality JOIN, with an
optional WHERE. Unsupported shapes return a structured warning.
"""

import re


_ID = r"[A-Za-z_][A-Za-z_0-9]*"
_TABLE = re.compile(rf"^({_ID})(?:\s+(?:AS\s+)?({_ID}))?$", re.I)
_JOIN = re.compile(r"\b(?:(LEFT|RIGHT|FULL|INNER)\s+(?:OUTER\s+)?)?JOIN\b", re.I)
_CLAUSES = ("FROM", "WHERE", "GROUP BY", "HAVING", "ORDER BY", "LIMIT", "QUALIFY")
_AGG = re.compile(r"\b(?:SUM|AVG|COUNT|MIN|MAX)\s*\(\s*(?!DISTINCT\b)", re.I)


def _clauses(sql):
    """Split at top-level clause keywords, ignoring strings and parentheses."""
    found = {}
    depth, quote, i = 0, None, 0
    while i < len(sql):
        char = sql[i]
        if quote:
            if char == quote:
                if i + 1 < len(sql) and sql[i + 1] == quote:
                    i += 2
                    continue
                quote = None
        elif char in ("'", '"'):
            quote = char
        elif char == "(":
            depth += 1
        elif char == ")":
            depth -= 1
        elif depth == 0:
            for name in _CLAUSES:
                if (sql[i:i + len(name)].upper() == name
                    and (i == 0 or not (sql[i - 1].isalnum() or sql[i - 1] == "_"))
                    and (i + len(name) == len(sql) or not (sql[i + len(name)].isalnum() or sql[i + len(name)] == "_"))):
                    found.setdefault(name, (i, i + len(name)))
                    i += len(name) - 1
                    break
        i += 1
    if depth != 0 or quote or "FROM" not in found:
        raise ValueError("Could not parse a top-level FROM clause")
    ordered = sorted(found.items(), key=lambda item: item[1][0])
    parts = {"SELECT": sql[len("SELECT"):found["FROM"][0]].strip()}
    for n, (name, (_, end)) in enumerate(ordered):
        stop = ordered[n + 1][1][0] if n + 1 < len(ordered) else len(sql)
        parts[name] = sql[end:stop].strip()
    return parts


def _table(text):
    match = _TABLE.fullmatch(text.strip())
    if not match:
        raise ValueError("Only plain table names with optional aliases are supported")
    name, alias = match.groups()
    if name.lower() not in ("customers", "orders"):
        raise ValueError("Only customers and orders are supported")
    return name, alias or name


def _from(text):
    join = _JOIN.search(text)
    if not join:
        name, alias = _table(text)
        return name, alias, None, {alias: name}, f"__review_base {alias}"
    base, alias = _table(text[:join.start()])
    tail = text[join.end():]
    on = re.search(r"\bON\b", tail, re.I)
    if not on or _JOIN.search(tail):
        raise ValueError("Only one JOIN with ON is supported")
    other, other_alias = _table(tail[:on.start()])
    condition = tail[on.end():].strip()
    if not re.fullmatch(rf"{_ID}\.{_ID}\s*=\s*{_ID}\.{_ID}", condition):
        raise ValueError("Only one qualified equality JOIN key is supported")
    kind = (join.group(1) or "INNER").upper()
    relation = f"__review_base {alias} {kind} JOIN {other} {other_alias} ON {condition}"
    return base, alias, kind, {alias: base, other_alias: other}, relation


def _nullable_references(con, aliases, predicate):
    references = []
    for alias, table in aliases.items():
        columns = con.execute('SELECT name, "notnull" FROM pragma_table_info(?)', [table]).fetchall()
        for column, not_null in columns:
            if not_null:
                continue
            qualified = re.search(rf"\b{re.escape(alias)}\.{re.escape(column)}\b", predicate, re.I)
            unqualified = re.search(rf"(?<![\w.]){re.escape(column)}\b", predicate, re.I)
            if qualified or (unqualified and sum(
                any(c.lower() == column.lower() for c, _ in con.execute(
                    'SELECT name, "notnull" FROM pragma_table_info(?)', [t]).fetchall())
                for t in aliases.values()) == 1):
                references.append((alias, column))
    return references


def review_query(con, sql):
    """Return structured warnings for one DuckDB SELECT; print nothing.

    The diagnostic statement scans the join once and counts both joined rows
    and distinct left-row IDs. Their difference detects fan-out even if an
    inner join also drops other left rows and the net count looks unchanged.
    The separate baseline count touches only the left table, not the join.

    A comparison with NULL evaluates to UNKNOWN; WHERE retains only TRUE.
    Thus a nullable filter can change the population and dashboard result.
    This is a correctness issue, not a SQL style preference.
    """
    if not isinstance(sql, str) or not re.match(r"^\s*SELECT\b", sql, re.I):
        raise ValueError("Expected one SELECT statement")
    sql = sql.strip().rstrip(";").strip()
    if ";" in sql:
        raise ValueError("Expected one SELECT statement")
    con.execute("EXPLAIN " + sql).fetchall()  # Validate syntax and free the result.
    try:
        parts = _clauses(sql)
        base, alias, join_kind, aliases, relation = _from(parts["FROM"])
    except ValueError as exc:
        return [{"code": "UNSUPPORTED_QUERY", "severity": "warning",
                 "message": str(exc), "details": {}}]

    predicate = parts.get("WHERE", "")
    nullable = _nullable_references(con, aliases, predicate) if predicate else []
    filter_sql = f" FILTER (WHERE ({predicate}))" if predicate else ""
    measures = [f"COUNT(*){filter_sql} AS joined_rows",
                f"COUNT(DISTINCT {alias}.__review_rowid){filter_sql} AS matched_base_rows"]
    for i, (a, column) in enumerate(nullable):
        measures.append(f"COUNT(*) FILTER (WHERE {a}.{column} IS NULL "
                        f"AND ({predicate}) IS NOT TRUE) AS null_{i}")
    # One aggregate query over the join; no second execution of the full join.
    diagnostic = (f"WITH __review_base AS (SELECT *, ROW_NUMBER() OVER () AS __review_rowid "
                  f"FROM {base}) SELECT {', '.join(measures)} FROM {relation}")
    values = con.execute(diagnostic).fetchall()[0]
    joined_rows, matched_base_rows = values[:2]
    fanout_rows = joined_rows - matched_base_rows

    # Only compare like populations when WHERE mentions the left table alone.
    right_aliases = [a for a in aliases if a != alias]
    base_only = not any(re.search(rf"\b{re.escape(a)}\.", predicate, re.I)
                        for a in right_aliases)
    baseline_sql = f"SELECT COUNT(*) FROM {base} {alias}"
    if predicate and base_only:
        baseline_sql += f" WHERE {predicate}"
    base_rows = con.execute(baseline_sql).fetchall()[0][0]

    warnings = []
    if join_kind and (joined_rows != base_rows or fanout_rows > 0):
        warnings.append({"code": "JOIN_ROW_COUNT_CHANGE", "severity": "warning",
                         "message": "Join changes the left-table row multiplicity.",
                         "details": {"base_rows": base_rows, "joined_rows": joined_rows,
                                     "matched_base_rows": matched_base_rows,
                                     "fanout_rows": fanout_rows}})
    for i, (a, column) in enumerate(nullable):
        if values[i + 2] > 0:
            warnings.append({"code": "NULL_FILTER", "severity": "warning",
                             "message": "WHERE drops rows with an unknown value.",
                             "details": {"column": f"{a}.{column}",
                                         "dropped_null_rows": values[i + 2]}})
    if join_kind and fanout_rows > 0 and _AGG.search(parts["SELECT"]):
        warnings.append({"code": "FANOUT_AGGREGATE", "severity": "warning",
                         "message": "Aggregate uses duplicated left-table rows; verify its grain.",
                         "details": {"fanout_rows": fanout_rows}})
    return warnings


In [ ]:
#Phase 2
"""Review simple DuckDB SELECTs over customers and orders.

Supported SQL: one plain base table, optionally one equality JOIN, with an
optional WHERE. Unsupported shapes return a structured warning.
"""

import re


_ID = r"[A-Za-z_][A-Za-z_0-9]*"
_TABLE = re.compile(rf"^({_ID})(?:\s+(?:AS\s+)?({_ID}))?$", re.I)
_JOIN = re.compile(r"\b(?:(LEFT|RIGHT|FULL|INNER)\s+(?:OUTER\s+)?)?JOIN\b", re.I)
_CLAUSES = ("FROM", "WHERE", "GROUP BY", "HAVING", "ORDER BY", "LIMIT", "QUALIFY")
_AGG = re.compile(r"\b(?:SUM|AVG|COUNT|MIN|MAX)\s*\(\s*(?!DISTINCT\b)", re.I)


def _clauses(sql):
    """Split at top-level clause keywords, ignoring strings and parentheses."""
    found = {}
    depth, quote, i = 0, None, 0
    while i < len(sql):
        char = sql[i]
        if quote:
            if char == quote:
                if i + 1 < len(sql) and sql[i + 1] == quote:
                    i += 2
                    continue
                quote = None
        elif char in ("'", '"'):
            quote = char
        elif char == "(":
            depth += 1
        elif char == ")":
            depth -= 1
        elif depth == 0:
            for name in _CLAUSES:
                if (sql[i:i + len(name)].upper() == name
                    and (i == 0 or not (sql[i - 1].isalnum() or sql[i - 1] == "_"))
                    and (i + len(name) == len(sql) or not (sql[i + len(name)].isalnum() or sql[i + len(name)] == "_"))):
                    found.setdefault(name, (i, i + len(name)))
                    i += len(name) - 1
                    break
        i += 1
    if depth != 0 or quote or "FROM" not in found:
        raise ValueError("Could not parse a top-level FROM clause")
    ordered = sorted(found.items(), key=lambda item: item[1][0])
    parts = {"SELECT": sql[len("SELECT"):found["FROM"][0]].strip()}
    for n, (name, (_, end)) in enumerate(ordered):
        stop = ordered[n + 1][1][0] if n + 1 < len(ordered) else len(sql)
        parts[name] = sql[end:stop].strip()
    return parts


def _table(text):
    match = _TABLE.fullmatch(text.strip())
    if not match:
        raise ValueError("Only plain table names with optional aliases are supported")
    name, alias = match.groups()
    if name.lower() not in ("customers", "orders"):
        raise ValueError("Only customers and orders are supported")
    return name, alias or name


def _from(text):
    join = _JOIN.search(text)
    if not join:
        name, alias = _table(text)
        return name, alias, None, {alias: name}, f"__review_base {alias}"
    base, alias = _table(text[:join.start()])
    tail = text[join.end():]
    on = re.search(r"\bON\b", tail, re.I)
    if not on or _JOIN.search(tail):
        raise ValueError("Only one JOIN with ON is supported")
    other, other_alias = _table(tail[:on.start()])
    condition = tail[on.end():].strip()
    if not re.fullmatch(rf"{_ID}\.{_ID}\s*=\s*{_ID}\.{_ID}", condition):
        raise ValueError("Only one qualified equality JOIN key is supported")
    kind = (join.group(1) or "INNER").upper()
    relation = f"__review_base {alias} {kind} JOIN {other} {other_alias} ON {condition}"
    return base, alias, kind, {alias: base, other_alias: other}, relation


def _nullable_references(con, aliases, predicate):
    references = []
    for alias, table in aliases.items():
        columns = con.execute('SELECT name, "notnull" FROM pragma_table_info(?)', [table]).fetchall()
        for column, not_null in columns:
            if not_null:
                continue
            qualified = re.search(rf"\b{re.escape(alias)}\.{re.escape(column)}\b", predicate, re.I)
            unqualified = re.search(rf"(?<![\w.]){re.escape(column)}\b", predicate, re.I)
            if qualified or (unqualified and sum(
                any(c.lower() == column.lower() for c, _ in con.execute(
                    'SELECT name, "notnull" FROM pragma_table_info(?)', [t]).fetchall())
                for t in aliases.values()) == 1):
                references.append((alias, column))
    return references


def review_query(con, sql):
    """Return structured warnings for one DuckDB SELECT; print nothing.

    The diagnostic statement scans the join once and counts both joined rows
    and distinct left-row IDs. Their difference detects fan-out even if an
    inner join also drops other left rows and the net count looks unchanged.
    The separate baseline count touches only the left table, not the join.

    A comparison with NULL evaluates to UNKNOWN; WHERE retains only TRUE.
    Thus a nullable filter can change the population and dashboard result.
    This is a correctness issue, not a SQL style preference.
    """
    if not isinstance(sql, str) or not re.match(r"^\s*SELECT\b", sql, re.I):
        raise ValueError("Expected one SELECT statement")
    sql = sql.strip().rstrip(";").strip()
    if ";" in sql:
        raise ValueError("Expected one SELECT statement")
    con.execute("EXPLAIN " + sql).fetchall()  # Validate syntax and free the result.
    try:
        parts = _clauses(sql)
        base, alias, join_kind, aliases, relation = _from(parts["FROM"])
    except ValueError as exc:
        return [{"code": "UNSUPPORTED_QUERY", "severity": "warning",
                 "message": str(exc), "details": {}}]

    predicate = parts.get("WHERE", "")
    nullable = _nullable_references(con, aliases, predicate) if predicate else []
    filter_sql = f" FILTER (WHERE ({predicate}))" if predicate else ""
    measures = [f"COUNT(*){filter_sql} AS joined_rows",
                f"COUNT(DISTINCT {alias}.__review_rowid){filter_sql} AS matched_base_rows"]
    for i, (a, column) in enumerate(nullable):
        measures.append(f"COUNT(*) FILTER (WHERE {a}.{column} IS NULL "
                        f"AND ({predicate}) IS NOT TRUE) AS null_{i}")
    # One aggregate query over the join; no second execution of the full join.
    diagnostic = (f"WITH __review_base AS (SELECT *, ROW_NUMBER() OVER () AS __review_rowid "
                  f"FROM {base}) SELECT {', '.join(measures)} FROM {relation}")
    values = con.execute(diagnostic).fetchall()[0]
    joined_rows, matched_base_rows = values[:2]
    fanout_rows = joined_rows - matched_base_rows

    # Only compare like populations when WHERE mentions the left table alone.
    right_aliases = [a for a in aliases if a != alias]
    base_only = not any(re.search(rf"\b{re.escape(a)}\.", predicate, re.I)
                        for a in right_aliases)
    baseline_sql = f"SELECT COUNT(*) FROM {base} {alias}"
    if predicate and base_only:
        baseline_sql += f" WHERE {predicate}"
    base_rows = con.execute(baseline_sql).fetchall()[0][0]

    warnings = []
    if join_kind and (joined_rows != base_rows or fanout_rows > 0):
        warnings.append({"code": "JOIN_ROW_COUNT_CHANGE", "severity": "warning",
                         "message": "Join changes the left-table row multiplicity.",
                         "details": {"base_rows": base_rows, "joined_rows": joined_rows,
                                     "matched_base_rows": matched_base_rows,
                                     "fanout_rows": fanout_rows}})
    for i, (a, column) in enumerate(nullable):
        if values[i + 2] > 0:
            warnings.append({"code": "NULL_FILTER", "severity": "warning",
                             "message": "WHERE drops rows with an unknown value.",
                             "details": {"column": f"{a}.{column}",
                                         "dropped_null_rows": values[i + 2]}})
    if join_kind and fanout_rows > 0 and _AGG.search(parts["SELECT"]):
        warnings.append({"code": "FANOUT_AGGREGATE", "severity": "warning",
                         "message": "Aggregate uses duplicated left-table rows; verify its grain.",
                         "details": {"fanout_rows": fanout_rows}})
    return warnings



*Your answer here.*

The reviewer caught the join’s duplicated customer rows before the $1,231.82 average could be treated as a customer-level metric. It also flagged the 600 observed rows excluded by region != 'West'. That is a measured loss, not a warning based solely on nullable schema metadata.

### Task 4.2: The one-page writeup (8 pts)

*Your answer here.*

Director, $812 is the right average per Wst customer. The roughly 1240 dashboad figure averages after joining orders, so customers with more orders count cultiple times.

We cannnot tell from West's higher revenue whether the programme is working. In a simulation of the effect of 25, the raw gap was 185.59, adjusting for prior account size brought it to 24.06.

Before presenting the result, we should define what each row represents, and check whether joings multiple customer and reconcile the result with Finance. A credible comparison is required to evaluate the programme.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* Oct.31, GPT-6 Sol

*Prompt:*

[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

*Reply:*

I wrote [review_query.py](/Users/zacklu/Documents/Codex/2026-11-01/sql-prep-act-as-an-analytics-3/outputs/review_query.py). It returns warning dictionaries with `code`, `severity`, `message`, and `details`; it prints nothing.

Running it against the notebook’s Phase 1 data gave:

| Query | Review result |
|---|---|
| `low_sql` | `[]` |
| `high_sql` | `JOIN_ROW_COUNT_CHANGE` and `FANOUT_AGGREGATE` |

**Yes, it would have caught the board figure.** The joined query averaged revenue over 19,082 rows representing 2,800 West customers, with 16,282 extra rows from fan-out. Its result was **$1,231.82**; the customer-level query returned **$811.17**.

The reviewer counts joined rows and distinct customer row IDs in one diagnostic join. I also verified that this detects fan-out when dropped rows make the *net* row count look unchanged. After the notebook introduces missing regions, it flags `region != 'West'` with `dropped_null_rows: 600`.

The reviewer is scoped to simple single-table or one-join `SELECT` queries over `customers` and `orders`. Unsupported query shapes receive an `UNSUPPORTED_QUERY` warning.

*Model and date:* Oct.31, GPT-6 Sol

*Prompt:*

[Prep] Act as an analytics engineer reviewing SQL before it reaches a dashboard.

[Request] Use my existing reviewer to assess Phase 2. Check whether it warns when a filter silently excludes customers with unknown regions and stays quiet when the corrected filter keeps them. Preserve its Phase 1 checks.

[Iterate] Use the data I already shared. Return structured warnings and report the actual row counts.

[Mechanism Check] Explain why the first filter loses customers with unknown regions and why the corrected filter keeps them. Distinguish a real loss of rows from a warning based only on a column being nullable.

[Evaluate] Review both Phase 2 filters and the window-function result. Verify the 1,046 and 1,646 filter counts, the 600 unknown regions, and the unchanged 4,000 window rows. Report what the reviewer caught, what it missed or falsely flagged, and an example that returned no warnings.


*Reply:*

I replayed the shared notebook’s data setup and ran the existing [review_query function](/Users/zacklu/Documents/Codex/2026-11-01/sql-prep-act-as-an-analytics-3/outputs/review_query.py) without changing it.

| Phase 2 query | Actual rows | Structured warnings |
|---|---:|---|
| `region != 'West'` | 1,046 | `[{"code": "NULL_FILTER", "severity": "warning", "message": "WHERE drops rows with an unknown value.", "details": {"column": "customers.region", "dropped_null_rows": 600}}]` |
| `region IS DISTINCT FROM 'West'` | 1,646 | `[]` |
| Window-function query | 4,000 | `[]` |

There are **600 customers with unknown (`NULL`) regions**. For them, `region != 'West'` evaluates to `UNKNOWN`, and `WHERE` keeps only rows for which the condition is `TRUE`. `IS DISTINCT FROM` treats `NULL` as distinct from `'West'`, so it keeps those 600 rows. The warning reports an **observed loss of 600**, rather than merely noting that `region` is nullable.

**Phase 1 checks remain intact:** the customer-level query returned `[]`; the joined query returned `JOIN_ROW_COUNT_CHANGE` and `FANOUT_AGGREGATE` for 19,082 joined rows versus 2,800 West customers.

The reviewer caught the Phase 2 row loss and gave no false warning for the corrected filter. It also returned no warnings for the window query, but that alone does **not** verify row preservation; I checked its result separately and confirmed **4,000 rows before and after**.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.